# Week 9, Exercise 7: Coding a Decoder-Only Transformer

We implement a small GPT-style language model from scratch and train it to predict the next character of a short text.

This notebook contains the full solution of Exercise 7 from the [Week 9 optional exercises](https://utm-csc413.github.io/2026F-website/optional_exercises/w09/exercises_w09.html). The saved outputs come from a run on a CPU; your numbers may differ slightly on Colab or with other PyTorch versions. Each training run takes a few seconds to about half a minute on a CPU.

## (a)-(c) Multi-head attention, the causal mask, the Transformer block, and the model

`MultiHeadAttention` takes separate query and key/value inputs, so that the same module can be used for cross-attention in Exercise 9. `Block` follows the encoder layer from the lecture: self-attention, Add & Norm, FFN, Add & Norm. The `use_mask` argument of `DecoderOnlyLM` is only there for part (e).

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_k = n_heads, d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):                       # (B, n, d_model) -> (B, h, n, d_k)
        B, n, _ = x.shape
        return x.view(B, n, self.h, self.d_k).transpose(1, 2)

    def forward(self, x_q, x_kv, mask=None):
        Q = self.split_heads(self.W_q(x_q))         # (B, h, n_q, d_k)
        K = self.split_heads(self.W_k(x_kv))        # (B, h, n_k, d_k)
        V = self.split_heads(self.W_v(x_kv))        # (B, h, n_k, d_k)
        scores = Q @ K.transpose(-2, -1) / self.d_k ** 0.5   # (B, h, n_q, n_k)
        if mask is not None:
            scores = scores + mask                  # mask: (n_q, n_k), 0 or -inf
        C = torch.softmax(scores, dim=-1) @ V       # (B, h, n_q, d_k)
        B, _, n_q, _ = C.shape
        C = C.transpose(1, 2).reshape(B, n_q, self.h * self.d_k)  # concatenate heads
        return self.W_o(C)

def causal_mask(n):
    return torch.triu(torch.full((n, n), float("-inf")), diagonal=1)

class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        a = self.norm1(x + self.attn(x, x, mask))   # self-attention, Add & Norm
        return self.norm2(a + self.ffn(a))          # FFN, Add & Norm

class DecoderOnlyLM(nn.Module):
    def __init__(self, vocab_size, max_len, d_model=64, n_heads=4, d_ff=128, n_layers=2):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, idx, use_mask=True):          # idx: (B, n) token indices
        n = idx.shape[1]
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(n))
        mask = causal_mask(n) if use_mask else None
        for block in self.blocks:
            x = block(x, mask)
        return self.out(x)                          # (B, n, vocab_size)

## (d) Training data

Each training example is a window of 32 characters `x`, and the targets `y` are the same window shifted by one character.

In [2]:
text = ("the quick brown fox jumps over the lazy dog. "
        "a lazy dog sleeps all day while the quick fox runs. ")
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text])
block_size = 32

def get_batch(batch_size=32):
    starts = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[s:s + block_size] for s in starts])
    y = torch.stack([data[s + 1:s + block_size + 1] for s in starts])   # x shifted by one
    return x, y

Every position in a window is trained to predict the next character at once. This is teacher forcing: the model always sees the true previous characters, never its own predictions. To generate, we repeatedly append the most likely next character.

In [3]:
def train(use_mask=True, steps=500):
    torch.manual_seed(0)
    model = DecoderOnlyLM(len(chars), block_size)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for step in range(steps):
        x, y = get_batch()
        logits = model(x, use_mask)
        loss = F.cross_entropy(logits.reshape(-1, len(chars)), y.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % 100 == 0 or step == steps - 1:
            print(f"step {step:3d}  loss {loss.item():.4f}")
    return model

@torch.no_grad()
def generate(model, prompt, n_new=60, use_mask=True):
    idx = torch.tensor([[stoi[c] for c in prompt]])
    for _ in range(n_new):
        logits = model(idx[:, -block_size:], use_mask)   # at most block_size tokens of context
        next_id = logits[0, -1].argmax()                 # most likely next character
        idx = torch.cat([idx, next_id.view(1, 1)], dim=1)
    return "".join(chars[i] for i in idx[0])

model = train()
print(generate(model, "the quick"))

step   0  loss 3.4381


step 100  loss 0.3380


step 200  loss 0.0674


step 300  loss 0.0486


step 400  loss 0.0507


step 499  loss 0.0444
the quick brown fox jumps over the lazy dog. a lazy dog sleeps all da


The training text is only about 100 characters long, so the model has essentially memorized it rather than learned English. Because greedy decoding is deterministic, it can also get stuck repeating the same phrase.

## (e) Training without the causal mask

In [4]:
model_nomask = train(use_mask=False)
print(generate(model_nomask, "the quick", use_mask=False))

step   0  loss 3.4474


step 100  loss 0.7331


step 200  loss 0.1094


step 300  loss 0.0155


step 400  loss 0.0065


step 499  loss 0.0036
the quick br quick r que ruick run rox r rox r qun fox r r a fov l fo


The training loss is even lower than with the mask, but generation produces garbage. Without the mask, position $i$ can attend to position $i+1$, which holds exactly the character it is supposed to predict, so the model learns to copy it from the input. At generation time, the next character is not in the input yet, so the copying strategy has nothing to copy. The causal mask makes training match generation: each position only sees the characters before it.